In [12]:
# from pathlib import Path
# from langchain_community.document_loaders import DirectoryLoader, TextLoader
# from langchain_text_splitters import (
#     MarkdownHeaderTextSplitter,
#     PythonCodeTextSplitter,
# )

# DOCS_DIR = Path("../../docs").resolve()

# def load_and_chunk_docs(docs_dir=DOCS_DIR):
#     # 1. Load Markdown Files
#     md_loader = DirectoryLoader(
#         str(docs_dir),
#         glob="**/*.md",
#         loader_cls=TextLoader,
#         loader_kwargs={"encoding": "utf-8"}
#     )
#     md_documents = md_loader.load()

#     # 2. Load Python Scripts
#     py_loader = DirectoryLoader(
#         str(docs_dir),
#         glob="**/*.py",
#         loader_cls=TextLoader,
#         loader_kwargs={"encoding": "utf-8"}
#     )
#     py_documents = py_loader.load()

#     all_chunks = []

#     # 3. Header-based Markdown Splitting
#     headers_to_split_on = [
#         ("#", "Header 1"),
#         ("##", "Header 2"),
#         ("###", "Header 3"),
#     ]
#     markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

#     for doc in md_documents:
#         chunks = markdown_splitter.split_text(doc.page_content)
#         for chunk in chunks:
#             chunk.metadata["source"] = doc.metadata.get("source", "")
#             chunk.metadata["file_type"] = "markdown"
#             all_chunks.append(chunk)

#     # 4. AST Syntax-Aware Python Splitting
#     # Set chunk_size high enough (~1500) so entire functions/classes remain intact
#     py_splitter = PythonCodeTextSplitter(
#         chunk_size=1500,
#         chunk_overlap=200
#     )

#     py_chunks = py_splitter.split_documents(py_documents)
#     for chunk in py_chunks:
#         chunk.metadata["file_type"] = "python"
#         all_chunks.append(chunk)

#     raw_documents = md_documents + py_documents
#     return raw_documents, all_chunks

In [13]:
import shutil
from pathlib import Path
from dotenv import load_dotenv

# LangChain document loaders & splitters
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    PythonCodeTextSplitter,
)

# Vector DB & Embedding imports
from langchain_chroma import Chroma
from langchain_community.embeddings import FastEmbedEmbeddings

load_dotenv()

# Resolve paths based on RAG_src/processing/ location
# .parents[0] = RAG_src/processing
# .parents[1] = RAG_src
# .parents[2] = Project Root (LUX_Data_Operations_Guide)
NOTEBOOK_DIR = Path(".").resolve()
ROOT_DIR = NOTEBOOK_DIR.parents[1]

DOCS_DIR = ROOT_DIR / "docs"
CHROMA_PATH = ROOT_DIR / "RAG_src" / "vectorDB" / "chroma_db"

print(f"Docs Directory: {DOCS_DIR}")
print(f"Target ChromaDB Directory: {CHROMA_PATH}")

Docs Directory: D:\Projects\LUX\LUX_Data_Operations_Guide\docs
Target ChromaDB Directory: D:\Projects\LUX\LUX_Data_Operations_Guide\RAG_src\vectorDB\chroma_db


In [14]:
def load_and_chunk_docs(docs_dir=DOCS_DIR):
    if not docs_dir.exists():
        raise FileNotFoundError(f"Docs path not found: {docs_dir}")

    # 1. Load Markdown Files
    md_loader = DirectoryLoader(
        str(docs_dir),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    md_documents = md_loader.load()

    # 2. Load Python Scripts
    py_loader = DirectoryLoader(
        str(docs_dir),
        glob="**/*.py",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    py_documents = py_loader.load()

    all_chunks = []

    # 3. Header-based Markdown Splitting
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    for doc in md_documents:
        chunks = markdown_splitter.split_text(doc.page_content)
        for chunk in chunks:
            chunk.metadata["source"] = doc.metadata.get("source", "")
            chunk.metadata["file_type"] = "markdown"
            all_chunks.append(chunk)

    # 4. AST Syntax-Aware Python Splitting
    py_splitter = PythonCodeTextSplitter(
        chunk_size=1500,
        chunk_overlap=200
    )

    py_chunks = py_splitter.split_documents(py_documents)
    for chunk in py_chunks:
        chunk.metadata["file_type"] = "python"
        all_chunks.append(chunk)

    raw_documents = md_documents + py_documents
    return raw_documents, all_chunks

In [15]:
# Execute document loading and splitting
raw_docs, chunks = load_and_chunk_docs()

print(f"Total raw documents loaded: {len(raw_docs)}")
print(f"Total chunks created: {len(chunks)}")

# Print unique processed file paths to confirm Python and Markdown files are captured
sources = sorted(list({chunk.metadata.get("source") for chunk in chunks}))
print("\nFiles included:")
for src in sources:
    print(f" - {src}")

Total raw documents loaded: 21
Total chunks created: 23

Files included:
 - D:\Projects\LUX\LUX_Data_Operations_Guide\docs\gemini-assistant\gemini-assistant.md
 - D:\Projects\LUX\LUX_Data_Operations_Guide\docs\index.md
 - D:\Projects\LUX\LUX_Data_Operations_Guide\docs\reference\knime\claims-p1\cmp-control.md
 - D:\Projects\LUX\LUX_Data_Operations_Guide\docs\src\utils\ingest.py
 - D:\Projects\LUX\LUX_Data_Operations_Guide\docs\t-ref.md
 - D:\Projects\LUX\LUX_Data_Operations_Guide\docs\tutorials\overview.md


In [16]:
import gc
import os
import shutil
import time

# Force memory cleanup to release file handles in the current Python kernel
gc.collect()


def remove_readonly_and_locked(func, path, exc_info):
    """Error handler to retry deleting files locked or marked read-only by Windows."""
    import stat

    os.chmod(path, stat.S_IWRITE)
    time.sleep(0.1)
    try:
        func(path)
    except Exception as e:
        print(f"Skipped deleting locked file: {path} ({e})")


if CHROMA_PATH.exists():
    print(f"Removing old vector database from {CHROMA_PATH}...")
    try:
        shutil.rmtree(CHROMA_PATH, onerror=remove_readonly_and_locked)
    except Exception as err:
        print(
            f"Warning: Could not fully delete directory due to active process locks: {err}"
        )

# Initialize the embedding function
embedding_function = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Generate embeddings and persist to disk
print("Generating embeddings and writing to ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function,
    persist_directory=str(CHROMA_PATH),
)

print(f"\nSuccess! Saved {len(chunks)} chunks to {CHROMA_PATH}")

Removing old vector database from D:\Projects\LUX\LUX_Data_Operations_Guide\RAG_src\vectorDB\chroma_db...
Skipped deleting locked file: D:\Projects\LUX\LUX_Data_Operations_Guide\RAG_src\vectorDB\chroma_db\df2869cf-3caa-4d1b-bc87-a0fcd7518740\data_level0.bin ([WinError 32] The process cannot access the file because it is being used by another process: 'D:\\Projects\\LUX\\LUX_Data_Operations_Guide\\RAG_src\\vectorDB\\chroma_db\\df2869cf-3caa-4d1b-bc87-a0fcd7518740\\data_level0.bin')
Skipped deleting locked file: D:\Projects\LUX\LUX_Data_Operations_Guide\RAG_src\vectorDB\chroma_db\df2869cf-3caa-4d1b-bc87-a0fcd7518740\header.bin ([WinError 32] The process cannot access the file because it is being used by another process: 'D:\\Projects\\LUX\\LUX_Data_Operations_Guide\\RAG_src\\vectorDB\\chroma_db\\df2869cf-3caa-4d1b-bc87-a0fcd7518740\\header.bin')
Skipped deleting locked file: D:\Projects\LUX\LUX_Data_Operations_Guide\RAG_src\vectorDB\chroma_db\df2869cf-3caa-4d1b-bc87-a0fcd7518740\length.b